# Praktikum Naïve Bayes
### 4233201 – Kecerdasan Buatan | Semester Ganjil TA 2026/2027
### Week/Sesi: 4/3 | Nama: Tika Altiora Sitanggang | NIM: 42324025

**Tujuan praktikum:** mahasiswa mampu menerapkan algoritma Naïve Bayes untuk klasifikasi.

## 0. Teori Singkat: Naïve Bayes
**Naïve Bayes** adalah algoritma klasifikasi berbasis probabilitas yang menggunakan **Teorema Bayes**
dengan asumsi bahwa semua fitur dalam dataset bersifat **independen** satu sama lain.

Teorema Bayes:

$$P(C \mid X) = \frac{P(X \mid C)\, P(C)}{P(X)}$$

dengan:
- $P(C \mid X)$: probabilitas kelas $C$ jika diberikan fitur $X$
- $P(X \mid C)$: probabilitas fitur $X$ terjadi jika kelasnya $C$
- $P(C)$: probabilitas awal (prior) kelas
- $P(X)$: probabilitas fitur $X$ di semua data

Terdapat tiga jenis utama Naïve Bayes, tergantung jenis data yang digunakan:

| Jenis | Cocok untuk data | Digunakan pada praktikum ini? |
|---|---|---|
| **Gaussian NB** | numerik kontinu, diasumsikan berdistribusi normal | **Ya** (dataset Iris) |
| **Multinomial NB** | data kategori berbasis frekuensi kemunculan (mis. teks) | Tidak |
| **Bernoulli NB** | data biner (0/1) | Tidak |

Karena dataset Iris berisi fitur numerik kontinu (panjang dan lebar sepal/petal), maka digunakan **Gaussian Naïve Bayes**.

**Isi laporan:**

| Bagian | Isi |
|---|---|
| 1 | Import library dan load dataset Iris |
| 2 | Encoding label dan split data train/test |
| 3 | Membangun dan melatih model Gaussian Naïve Bayes |
| 4 | Evaluasi akurasi model |
| 5 | Prediksi data baru (contoh dari modul) |
| 6 | **Tugas**: prediksi data baru `[7.3, 3.8, 4.6, 2.3]` |
| 7 | Kesimpulan |

# BAGIAN 1. Import Library dan Load Dataset

Dataset yang digunakan adalah **Iris** yang diambil langsung dari Scikit-Learn. Dataset ini terdiri atas
150 data bunga dengan **4 fitur numerik kontinu** (sepal length, sepal width, petal length, petal width)
dan **3 kelas spesies** (setosa, versicolor, virginica).

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

# Load dataset Iris dari scikit-learn
iris = load_iris()

## 1.1 Konversi Data ke DataFrame dan Pemisahan Fitur (X) dengan Label (y)
Data dikonversi menjadi **DataFrame Pandas** agar lebih mudah diolah. Kolom `Species` berisi nama
spesies sebagai label. Variabel `X` menyimpan keempat fitur, sedangkan `y` menyimpan label (target).

In [2]:
# Konversi data menjadi DataFrame Pandas
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["Species"] = iris.target_names[iris.target]

# Definisikan X (fitur) dan y (target/label)
X = df.drop(columns=["Species"])
y = df["Species"]

print(X)

     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                  5.1               3.5                1.4               0.2
1                  4.9               3.0                1.4               0.2
2                  4.7               3.2                1.3               0.2
3                  4.6               3.1                1.5               0.2
4                  5.0               3.6                1.4               0.2
..                 ...               ...                ...               ...
145                6.7               3.0                5.2               2.3
146                6.3               2.5                5.0               1.9
147                6.5               3.0                5.2               2.0
148                6.2               3.4                5.4               2.3
149                5.9               3.0                5.1               1.8

[150 rows x 4 columns]


In [3]:
y

0         setosa
1         setosa
2         setosa
3         setosa
4         setosa
         ...    
145    virginica
146    virginica
147    virginica
148    virginica
149    virginica
Name: Species, Length: 150, dtype: str

Dataset terdiri atas **150 baris** dan **4 kolom fitur**. Setiap spesies memiliki 50 data, sehingga kelasnya seimbang (balanced).

Dataset Iris berisi fitur-fitur numerik kontinu, yaitu:
- Sepal Length (Panjang kelopak)
- Sepal Width (Lebar kelopak)
- Petal Length (Panjang mahkota)
- Petal Width (Lebar mahkota)

Karena semua fitur ini berupa data kontinu, maka digunakan **Gaussian Naïve Bayes**.

# BAGIAN 2. Encoding Label dan Split Data Train-Test

## 2.1 Encoding Label
Label `Species` berupa teks (string), sehingga diubah menjadi angka (numerik) menggunakan
**LabelEncoder** dari Scikit-Learn. Urutannya mengikuti alfabet: `setosa = 0`, `versicolor = 1`, `virginica = 2`.

In [4]:
from sklearn.preprocessing import LabelEncoder

lab = LabelEncoder()
y = lab.fit_transform(y)
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [5]:
# Cek pemetaan angka ke nama spesies
print(dict(zip(range(3), lab.classes_)))

{0: 'setosa', 1: 'versicolor', 2: 'virginica'}


## 2.2 Split Data Train dan Test
Data dibagi menjadi **80% data latih (train)** dan **20% data uji (test)** dengan `random_state=0`
supaya pembagiannya tetap sama setiap kali dijalankan. Hasilnya: 120 data latih dan 30 data uji.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(X, test_size=0.2, random_state=0)
y_train, y_test = train_test_split(y, test_size=0.2, random_state=0)

print("Jumlah data latih:", len(X_train))
print("Jumlah data uji  :", len(X_test))

Jumlah data latih: 120
Jumlah data uji  : 30


# BAGIAN 3. Membangun dan Melatih Model Gaussian Naïve Bayes

`GaussianNB` diimpor dari Scikit-Learn karena fitur pada dataset Iris bersifat numerik kontinu.
Model dilatih (`fit`) menggunakan data latih `X_train` dan `y_train`.

In [7]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
NB_model = model.fit(X_train, y_train)

# BAGIAN 4. Prediksi dan Evaluasi Akurasi

In [8]:
result = NB_model.predict(X_test)
result

array([2, 1, 0, 2, 0, 2, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 2, 1,
       0, 0, 2, 0, 0, 1, 1, 0])

In [9]:
from sklearn.metrics import accuracy_score

print('Accuracy of data testing: ', accuracy_score(y_test, result) * 100)

Accuracy of data testing:  96.66666666666667


Akurasi model Gaussian Naïve Bayes pada dataset Iris adalah **96,67%**. Dari 30 data uji, **29 diprediksi
benar** dan hanya **1 yang salah**, yaitu data ke-135 (`sepal length=6.1, sepal width=2.6, petal length=5.6,
petal width=1.4`) yang sebenarnya `virginica` tetapi diprediksi `versicolor`. Ini wajar karena kedua spesies
tersebut memang memiliki ukuran petal yang saling berdekatan.

# BAGIAN 5. Prediksi Data Baru (Contoh dari Modul)

Dicoba prediksi untuk data baru `[5.1, 3.5, 1.4, 0.2]`. Jika input diberikan sebagai **list/array biasa**
(tanpa nama kolom), muncul **`UserWarning`** karena model dilatih menggunakan `DataFrame` yang punya nama
kolom (`iris.feature_names`), sedangkan array biasa tidak punya nama kolom sehingga strukturnya tidak
konsisten dengan data latih.

Solusinya: ubah data input menjadi **DataFrame** dengan nama kolom yang sama seperti data latih, supaya
tidak muncul peringatan.

In [10]:
# input_data sebagai DataFrame dengan nama kolom yang sama seperti data latih
input_data = pd.DataFrame(np.array([[5.1, 3.5, 1.4, 0.2]]), columns=iris.feature_names)

result = NB_model.predict(input_data)
prediction_label = lab.inverse_transform(result)
print('Hasil prediksi untuk input baru adalah:', prediction_label)

Hasil prediksi untuk input baru adalah: ['setosa']


Hasil prediksi untuk data `[5.1, 3.5, 1.4, 0.2]` adalah **setosa**, tanpa muncul `UserWarning` karena input
sudah berbentuk DataFrame dengan nama kolom yang sesuai.

# BAGIAN 6. Tugas: Prediksi Data Baru

Prediksi data baru berikut masuk ke species apa:

`input_data = [7.3, 3.8, 4.6, 2.3]`

Data input dibuat sebagai **DataFrame** dengan nama kolom yang sama dengan data latih, mengikuti solusi
pada Bagian 5, supaya tidak memunculkan `UserWarning`.

In [11]:
input_data = pd.DataFrame(np.array([[7.3, 3.8, 4.6, 2.3]]), columns=iris.feature_names)

result = NB_model.predict(input_data)
prediction_label = lab.inverse_transform(result)
print('Hasil prediksi untuk input baru adalah:', prediction_label)

Hasil prediksi untuk input baru adalah: ['virginica']


### Bukti: Probabilitas Tiap Kelas
Kode berikut menampilkan probabilitas (posterior) untuk tiap kelas menggunakan `predict_proba`, sehingga
keyakinan model terhadap hasil prediksi bisa diperiksa langsung.

In [12]:
proba = NB_model.predict_proba(input_data)

tabel_proba = pd.DataFrame(proba, columns=lab.classes_)
tabel_proba

,setosa,versicolor,virginica
0,4.830485e-194,3.680815e-07,1.0


### Hasil dan Analisis Tugas
| Input `[sepal length, sepal width, petal length, petal width]` | **Prediksi** |
|---|---|
| `[7.3, 3.8, 4.6, 2.3]` | **virginica** |

Probabilitas kelas `virginica` bernilai **1,0 (100%)**, sedangkan `setosa` dan `versicolor` bernilai 0.
Artinya model **sangat yakin** dengan prediksinya. Hal ini masuk akal karena nilai `sepal length` (7,3) dan
`petal width` (2,3) pada data ini tergolong besar, sesuai karakteristik bunga **virginica** yang umumnya
memiliki ukuran sepal dan petal paling besar di antara ketiga spesies.

# BAGIAN 7. Kesimpulan

1. **Naïve Bayes** mengklasifikasikan data berdasarkan **Teorema Bayes** dengan asumsi semua fitur
   bersifat independen. Karena fitur dataset Iris berupa numerik kontinu, digunakan **Gaussian Naïve Bayes**.
2. Model dilatih pada 120 data latih dan diuji pada 30 data uji, menghasilkan akurasi **96,67%**
   (29 dari 30 data uji benar).
3. Memberi input sebagai array biasa memunculkan `UserWarning` karena tidak memiliki nama kolom seperti
   data latih. Solusinya adalah mengubah input menjadi **DataFrame** dengan nama kolom yang sama.
4. Hasil **Tugas**: data baru `[7.3, 3.8, 4.6, 2.3]` diprediksi sebagai spesies **virginica**, dengan
   probabilitas 100%.